In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd
df = pd.read_csv(path + "/Q3_data.csv")


In [ ]:
df.head()


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.columns


In [ ]:
target_col = "Target"


X = df.drop(columns=[target_col])
y = df[target_col]


num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(exclude=["number"]).columns

X[num_cols] = X[num_cols].fillna(X[num_cols].median())

if len(cat_cols) > 0:
    X[cat_cols] = X[cat_cols].fillna(X[cat_cols].mode().iloc[0])

print("Missing values after handling:", X.isna().sum().sum())
print(X.shape)
print(y.shape)
y.value_counts()


In [ ]:
dup_count = df.duplicated().sum()
print("Number of duplicate rows:", dup_count)

if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates removed ")
else:
    print("No duplicates found ")

In [ ]:
cat_cols = X.select_dtypes(exclude=["number"]).columns
print("Categorical columns:", list(cat_cols))

if len(cat_cols) > 0:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    print("Categorical columns encoded using one-hot encoding.")
else:
    print("No categorical columns found, encoding not needed.")

print("X shape after encoding:", X.shape)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaling done.")
print("X_scaled shape:", X_scaled.shape)


In [ ]:
import pandas as pd

counts = y.value_counts()
ratios = (counts / counts.sum()).round(4)

print("Target counts:\n", counts)
print("\nTarget ratios:\n", ratios)

minority_ratio = ratios.min()

if minority_ratio < 0.40:
    print("\nTarget is IMBALANCED.")
else:
    print("\nTarget is NOT imbalanced (roughly balanced).")


In [ ]:
%pip install CatBoost

In [ ]:
X = df.drop(columns=["Target"])
y = df["Target"]


In [ ]:
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score


X_use = X_scaled

class_ratios = y.value_counts(normalize=True)
is_imbalanced = class_ratios.min() < 0.40

if is_imbalanced:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    metric_name = "F1-score"
else:
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    metric_name = "Accuracy"

scores = []

for train_idx, val_idx in cv.split(X_use, y):
    X_train, X_val = X_use[train_idx], X_use[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        verbose=0,
        random_state=42
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    if is_imbalanced:
        score = f1_score(y_val, preds, average="binary")
    else:
        score = accuracy_score(y_val, preds)

    scores.append(score)

print(f"Metric used: {metric_name}")
print(f"Scores per fold: {np.round(scores, 4)}")
print(f"Average {metric_name}: {np.mean(scores):.4f}")


In [ ]:
from catboost import CatBoostClassifier
import pandas as pd
import matplotlib.pyplot as plt

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    verbose=0,
    random_state=42
)

model.fit(X, y)

importances = model.get_feature_importance()
feat_imp = pd.DataFrame({
    "feature": X.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feat_imp["feature"].head(20)[::-1], feat_imp["importance"].head(20)[::-1])
plt.xlabel("Importance")
plt.title("Top 20 Feature Importances (CatBoost)")
plt.show()


In [ ]:
golden_feature = feat_imp.iloc[0]["feature"]
golden_importance = feat_imp.iloc[0]["importance"]

print("Golden Feature:", golden_feature)
print("Importance Score:", golden_importance)


In [ ]:
importances = model.get_feature_importance()
feat_imp = pd.DataFrame({
    "feature": X.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

golden_feature = feat_imp.iloc[0]["feature"]

print("Golden Feature:", golden_feature)


In [ ]:
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

X_golden = X[[golden_feature]]

class_ratios = y.value_counts(normalize=True)
is_imbalanced = class_ratios.min() < 0.40

if is_imbalanced:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    metric_name = "F1-score"
else:
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    metric_name = "Accuracy"

full_scores = []
for train_idx, val_idx in cv.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_full = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        verbose=0,
        random_state=42
    )
    model_full.fit(X_train, y_train)
    preds = model_full.predict(X_val)

    score = f1_score(y_val, preds) if is_imbalanced else accuracy_score(y_val, preds)
    full_scores.append(score)

golden_scores = []
for train_idx, val_idx in cv.split(X_golden, y):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_golden = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        verbose=0,
        random_state=42
    )
    model_golden.fit(X_train, y_train)
    preds = model_golden.predict(X_val)

    score = f1_score(y_val, preds) if is_imbalanced else accuracy_score(y_val, preds)
    golden_scores.append(score)

print(f"Metric used: {metric_name}\n")

print("Full model scores per fold:", np.round(full_scores, 4))
print(f"Full model average {metric_name}: {np.mean(full_scores):.4f}\n")

print("Golden feature scores per fold:", np.round(golden_scores, 4))
print(f"Golden feature average {metric_name}: {np.mean(golden_scores):.4f}\n")

diff = np.mean(full_scores) - np.mean(golden_scores)
print(f"Difference (Full - Golden): {diff:.4f}")
